# LifeMate — train a tool-use adapter

Private train data only. This notebook prepares causal trajectories and runs QLoRA on the pinned Qwen base. The default is a **one-step GPU smoke test**, not a full fine-tune or a performance claim. Run a full epoch only after checking retained examples, memory use and training loss.


In [ ]:
REPO_URL = "https://github.com/AmnO-O/ZolaAIC.git"
REVISION = "main"  # Pin a commit for reproducibility.
REPO_DIR = "/kaggle/working/ZolaAIC"
TRAIN_DIR = "/kaggle/input/datasets/amnpham/zalochallenge/train+public_test/train"
MODEL_PATH = ""  # Existing pinned LOCAL base model directory, or download below.
MAX_LENGTH = 8192
MAX_STEPS = 1  # Smoke test only. Use -1 for a full training epoch after this succeeds.
EPOCHS = 1


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

repo = Path(REPO_DIR).resolve()
if repo.exists():
    actual_root = Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=repo, text=True).strip()).resolve()
    if actual_root != repo:
        raise RuntimeError("REPO_DIR must point to the repository root")
    remote = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
    if remote != REPO_URL:
        raise RuntimeError(f"Unexpected repository at {repo}; choose a new REPO_DIR")
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=repo, text=True).strip()
    if dirty:
        raise RuntimeError("Checkout has local edits. Preserve them and choose a new REPO_DIR.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)

# Works for a remote branch, tag, or exact commit; never merges local changes.
subprocess.run(["git", "fetch", "origin", REVISION], cwd=repo, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo, check=True)
if not (repo / "src/lifemate/prompt_assets/system.txt").is_file():
    raise RuntimeError("This revision predates the clean pipeline; select its notebook or a newer revision.")

COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print("Source commit:", COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements-train.txt")], check=True)


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


In [ ]:
import os
import uuid
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError("Select a Kaggle GPU before training")
train_dir = Path(TRAIN_DIR)
if not (train_dir / "train.jsonl").is_file():
    raise FileNotFoundError("Set TRAIN_DIR to the train split; public/private test is forbidden")
if not MODEL_PATH:
    subprocess.run([sys.executable, "scripts/download_models.py", "--component", "llm"], cwd=repo, check=True)
model_path = Path(MODEL_PATH).resolve() if MODEL_PATH else repo / "models" / "qwen3.5-4b"
run_dir = Path("/kaggle/working/training") / uuid.uuid4().hex[:12]
run_dir.mkdir(parents=True)
data_dir = run_dir / "data"
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8")
subprocess.run([sys.executable, "scripts/prepare_sft.py", "--train-dir", str(train_dir),
                "--output", str(data_dir)], cwd=repo, check=True)
manifest = json.loads((data_dir / "manifest.json").read_text())
print("Train / validation samples:", len(manifest["train"]), len(manifest["validation"]))
print("Rejected trajectories:", len(manifest["rejected"]))
print("Output:", run_dir)


In [ ]:
command = [sys.executable, "scripts/train_adapter.py", "--model", str(model_path),
           "--data", str(data_dir), "--output", str(run_dir / "training"),
           "--max-length", str(MAX_LENGTH), "--epochs", str(EPOCHS), "--max-steps", str(MAX_STEPS)]
if MAX_STEPS == 1:
    command.append("--smoke")
with (run_dir / "training.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, env={**os.environ, "PYTHONUNBUFFERED":"1"})
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
    returncode = process.wait()
if returncode:
    raise RuntimeError("Training failed; inspect training.log and skipped_length.json")
print("Adapter:", run_dir / "training" / "adapter")
print("Validation loss is not tool-use accuracy. A one-step smoke adapter is not a trained solution.")


## Evaluate the frozen adapter

In the inference notebook set `ADAPTER_PATH` to the local adapter directory and `IDS_FILE` to this run’s `data/manifest.json`; the runner selects its validation IDs. Install `requirements-train.txt` when loading an adapter. First use train reference transcripts, then real audio with the same frozen adapter. Compare against the unadapted base on identical validation IDs/settings. Do not use validation errors for repeated prompt tuning. Validation loss alone does not establish task success.

Keep generated examples, logs, checkpoints and notebook outputs private. The inference backend loads the base and adapter locally, merges the adapter, then uses the same agent. There are no teacher models or training calls at inference.
